In [1]:
# =====================================================
# Version 2: Gradient Boosting House Price Prediction
# =====================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error


# =====================================================
# 1. 원본 데이터 불러오기
# =====================================================

train_raw = pd.read_csv('modulabs-di/train.csv')
test_raw = pd.read_csv('modulabs-di/test.csv')

print("Train:", train_raw.shape)
print("Test:", test_raw.shape)

Train: (1168, 81)
Test: (292, 80)


In [2]:
# =====================================================
# 2. 새로운 변수 만들기 (Feature Engineering)
# =====================================================

def make_features(df):
    df = df.copy()

    # 집 전체 면적
    df['TotalSF'] = (
        df['TotalBsmtSF'] +
        df['1stFlrSF'] +
        df['2ndFlrSF']
    )

    # 집 전체 욕실 수
    df['TotalBath'] = (
        df['FullBath'] +
        0.5 * df['HalfBath'] +
        df['BsmtFullBath'] +
        0.5 * df['BsmtHalfBath']
    )

    # 집의 전체 현관/베란다 면적
    df['TotalPorchSF'] = (
        df['OpenPorchSF'] +
        df['EnclosedPorch'] +
        df['3SsnPorch'] +
        df['ScreenPorch']
    )

    # 집을 지은 후 경과 연수
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']

    # 마지막 리모델링 이후 경과 연수
    df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']

    # 집의 품질과 거주 면적을 결합
    df['QualityArea'] = (
        df['OverallQual'] * df['GrLivArea']
    )

    return df


# 정답 변수와 ID 분리
y = train_raw['SalePrice']
test_ids = test_raw['Id']

X = train_raw.drop(columns=['SalePrice', 'Id'])
X_test = test_raw.drop(columns=['Id'])

# 새로운 변수 생성
X = make_features(X)
X_test = make_features(X_test)

In [3]:
# =====================================================
# 3. 결측치 처리
# =====================================================

# 시설이 없는 경우를 나타내는 범주형 컬럼
none_cols = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
    'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual',
    'GarageCond', 'PoolQC', 'Fence', 'MiscFeature',
    'MasVnrType'
]

# 시설이 없는 경우 0으로 처리할 수치형 컬럼
zero_cols = [
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
    'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath',
    'GarageYrBlt', 'GarageCars', 'GarageArea',
    'MasVnrArea'
]

for col in none_cols:
    if col in X.columns:
        X[col] = X[col].fillna('None')
        X_test[col] = X_test[col].fillna('None')

for col in zero_cols:
    if col in X.columns:
        X[col] = X[col].fillna(0)
        X_test[col] = X_test[col].fillna(0)

In [4]:

# =====================================================
# 4. 수치형 / 범주형 컬럼 구분
# =====================================================

categorical_cols = X.select_dtypes(
    include=['object', 'category']
).columns.tolist()

numeric_cols = X.select_dtypes(
    include='number'
).columns.tolist()

C:\Users\chaee\AppData\Local\Temp\ipykernel_3356\1426985382.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


In [5]:
# =====================================================
# 5. 전처리 설정
# =====================================================

# 범주형: 남은 결측치 최빈값 처리 + 원 핫 인코딩
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    ))
])

# 수치형: 남은 결측치 중앙값 처리
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

preprocessor = ColumnTransformer([
    ('cat', categorical_pipeline, categorical_cols),
    ('num', numeric_pipeline, numeric_cols)
])


In [6]:

# =====================================================
# 6. 모델 설정
# =====================================================

model = Pipeline([
    ('preprocessor', preprocessor),

    ('regressor', GradientBoostingRegressor(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=2,
        min_samples_leaf=3,
        subsample=0.8,
        max_features=0.8,
        loss='huber',
        random_state=42
    ))
])


In [7]:
# =====================================================
# 7. 학습 데이터 / 검증 데이터 분리
# =====================================================

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# =====================================================
# 8. 집값 로그 변환
# =====================================================

y_train_log = np.log1p(y_train)


# =====================================================
# 9. 모델 학습
# =====================================================

model.fit(X_train, y_train_log)


# =====================================================
# 10. 검증 데이터 예측
# =====================================================

valid_pred_log = model.predict(X_valid)

# 로그 가격을 실제 가격으로 되돌리기
valid_pred = np.expm1(valid_pred_log)

# 음수 가격 방지
valid_pred = np.maximum(valid_pred, 0)

rmse = np.sqrt(
    mean_squared_error(y_valid, valid_pred)
)

print(f"Version 2 Validation RMSE: {rmse:.2f}")


Version 2 Validation RMSE: 20976.10


In [8]:
# =====================================================
# 11. 전체 학습 데이터로 다시 학습
# =====================================================

model.fit(X, np.log1p(y))

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](85,)","['MSSubClass','MSZoning','LotFrontage',...,'HouseAge','RemodAge', 'QualityArea']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,85
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subs

In [9]:
# =====================================================
# 12. 테스트 데이터 예측
# =====================================================

test_pred_log = model.predict(X_test)

test_pred = np.expm1(test_pred_log)
test_pred = np.maximum(test_pred, 0)

In [11]:
# =====================================================
# 13. 제출 파일 만들기
# =====================================================

submission_v2 = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_pred
})

submission_v2.to_csv(
    'submission_v2.csv',
    index=False
)

print("\n제출 파일 생성 완료!")
print(submission_v2.head())
print("파일 크기:", submission_v2.shape)


제출 파일 생성 완료!
    Id      SalePrice
0   86  285786.551906
1  227  276912.494155
2  492  156748.179801
3   64  140722.262829
4  576  117834.947239
파일 크기: (292, 2)
